In [ ]:
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

df = pd.read_parquet("../data/processed/train_merged.parquet")

# 1. Missingness by feature family (the dataset's column names encode families)
patterns = {
    "C": r"^C\d+$", "D": r"^D\d+$", "M": r"^M\d+$", "V": r"^V\d+$",
    "card": r"^card\d$", "addr": r"^addr\d$", "dist": r"^dist\d$", "id": r"^id_\d+$",
}


def family(col):
    for name, pat in patterns.items():
        if re.match(pat, col):
            return name
    return "other"


miss = df.isna().mean()
fam = pd.DataFrame({"missing": miss, "family": miss.index.map(family)})
print(fam.groupby("family")["missing"].agg(["size", "mean", "max"]).round(3))

# 2. Does "has identity record" carry signal? And fraud rate by product type
df["has_identity"] = df["id_01"].notna()
print(df.groupby("has_identity")["isFraud"].agg(["mean", "size"]).round(4))
print(df.groupby("ProductCD")["isFraud"].agg(["mean", "size"]).round(4))

# 3. Amount distribution by class, and fraud rate by week
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for label, s in df.groupby("isFraud")["TransactionAmt"]:
    ax[0].hist(np.log1p(s), bins=60, alpha=0.5, density=True, label=f"isFraud={label}")
ax[0].set_xlabel("log1p(TransactionAmt)")
ax[0].legend()
df.groupby(df["TransactionDT"] // (86400 * 7))["isFraud"].mean().plot(ax=ax[1])
ax[1].set_xlabel("week")
ax[1].set_ylabel("fraud rate")
plt.tight_layout()
plt.savefig("../docs/eda_amount_and_weekly_fraud.png", dpi=120)
plt.show()